In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed

sys.path.append(os.path.join('..', 'scripts'))

from utils.config import results_folder, data_path
from utils.dataset import k_fold_trainval_test_multi_object_styles

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

START_SEED = 50
END_SEED = 200
SEEDS = range(START_SEED, END_SEED)
FOLDER_NAME = '_deferral'

SCRIPTS = [
    'scout_proper',
    'scout_no_decouple',
    'knn_script',
    'mean_script',
    'mlp_script',
    'linear_regression_script',
    'mf_script',
    'oracle_script',
]

METHODS = ['Hunyuan3D', 'InstantMesh', 'Trellis', 'TripoSR']
METHOD_MARKERS = ['o', 's', 'D', '^']
METHOD_COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']

EXPERIMENT_TYPES = ['memories', 'latency_memories', 'latencies']

In [ ]:
# ---------------------------------------------------------------------------
# Pareto frontier functions
# ---------------------------------------------------------------------------

def _deduplicate_x(xs, ys):
    """For points sharing the same x, keep only the one with the highest y.

    Args:
        xs: Array of x coordinates.
        ys: Array of y coordinates.

    Returns:
        tuple: (unique_xs, best_ys) as numpy arrays.
    """
    xs, ys = np.array(xs), np.array(ys)
    unique_xs = np.unique(xs)
    best_ys = np.array([ys[xs == ux].max() for ux in unique_xs])
    return unique_xs, best_ys


def _pareto_frontier(xs, ys):
    """Compute the Pareto frontier from a set of points.

    Assumes lower x is better, higher y is better. For tied x values,
    only the point with the highest y is kept. Returns frontier points
    sorted by ascending x.

    Args:
        xs: Array of x coordinates.
        ys: Array of y coordinates.

    Returns:
        tuple: (frontier_x, frontier_y) as numpy arrays.
    """
    deduped_xs, deduped_ys = _deduplicate_x(xs, ys)

    order = np.argsort(deduped_xs)
    sorted_xs, sorted_ys = deduped_xs[order], deduped_ys[order]

    frontier_x, frontier_y = [], []
    best_y = -np.inf
    for x, y in zip(sorted_xs, sorted_ys):
        if y > best_y:
            best_y = y
            frontier_x.append(x)
            frontier_y.append(y)

    return np.array(frontier_x), np.array(frontier_y)


def _pareto_area(points_x, points_y, ref_x=None, ref_y=None, last_x=None,
                 first_x=None):
    """Compute area under the Pareto frontier (step function).

    Assumes higher y is better and lower x is better.

    Args:
        points_x: X coordinates (cost axis).
        points_y: Y coordinates (utility axis).
        ref_x: Reference x bound (rightmost). Defaults to max of pareto_x.
        ref_y: Reference y bound (bottom). Defaults to 0.
        last_x: If set, extend the frontier rightward to this x.
        first_x: If set, extend the frontier leftward to this x with y=0.

    Returns:
        tuple: (area, pareto_x, pareto_y).
    """
    pareto_x, pareto_y = _pareto_frontier(points_x, points_y)

    if first_x is not None and pareto_x[0] > first_x:
        pareto_x = np.insert(pareto_x, 0, first_x)
        pareto_y = np.insert(pareto_y, 0, 0)
    if last_x is not None and pareto_x[-1] < last_x:
        pareto_x = np.append(pareto_x, last_x)
        pareto_y = np.append(pareto_y, pareto_y[-1])

    if ref_y is None:
        ref_y = 0
    if ref_x is None:
        ref_x = pareto_x[-1]

    area = 0.0
    for i in range(len(pareto_x)):
        x_right = pareto_x[i + 1] if i + 1 < len(pareto_x) else ref_x
        height = pareto_y[i] - ref_y
        width = x_right - pareto_x[i]
        area += height * width

    return area, pareto_x, pareto_y


def _pareto_area_trapezoidal(points_x, points_y, ref_y=None, last_x=None,
                              first_x=None):
    """Compute area under the Pareto frontier using trapezoidal rule.

    Args:
        points_x: X coordinates (cost axis).
        points_y: Y coordinates (utility axis).
        ref_y: Reference y bound (bottom). Defaults to 0.
        last_x: If set, extend the frontier rightward to this x.
        first_x: If set, extend the frontier leftward to this x with y=0.

    Returns:
        tuple: (area, pareto_x, pareto_y).
    """
    pareto_x, pareto_y = _pareto_frontier(points_x, points_y)

    if first_x is not None and pareto_x[0] > first_x:
        pareto_x = np.insert(pareto_x, 0, first_x)
        pareto_y = np.insert(pareto_y, 0, 0)
    if last_x is not None and pareto_x[-1] < last_x:
        pareto_x = np.append(pareto_x, last_x)
        pareto_y = np.append(pareto_y, pareto_y[-1])

    if ref_y is None:
        ref_y = 0

    area = 0.0
    for i in range(len(pareto_x) - 1):
        width = pareto_x[i + 1] - pareto_x[i]
        height = ((pareto_y[i] - ref_y) + (pareto_y[i + 1] - ref_y)) / 2
        area += height * width

    return area, pareto_x, pareto_y


# ---------------------------------------------------------------------------
# Area computation helpers
# ---------------------------------------------------------------------------

def _extract_pareto_points(memory_utilities, min_utility_mean, max_utility_mean):
    """Extract Pareto-plottable (x, y) from a memory_utilities array.

    Args:
        memory_utilities: Array of shape (n_points, channels, n_methods+1).
        min_utility_mean: Lower bound for normalization.
        max_utility_mean: Upper bound for normalization.

    Returns:
        tuple: (points_x, points_y, last_x, first_x).
    """
    points_x, points_y = [], []
    for i in range(memory_utilities.shape[0]):
        points_x.append(memory_utilities[i, -1, -1])
        raw_value = memory_utilities[i, 0, -1]
        points_y.append((raw_value - min_utility_mean) / (max_utility_mean - min_utility_mean))
    last_x = np.max(memory_utilities[0, 1, :-1])
    first_x = np.min(points_x)
    return points_x, points_y, last_x, first_x


def _compute_areas_for_experiment(which_exp, test_datasets, scripts, seeds,
                                  start_seed, zero_max_utility=False):
    """Compute normalized Pareto areas across seeds for each script.

    Args:
        which_exp: Experiment type ('memories', 'latencies', 'latency_memories').
        test_datasets: List of y_test arrays, one per seed.
        scripts: List of script names.
        seeds: Range of seed values.
        start_seed: First seed (for indexing into test_datasets).
        zero_max_utility: If True, set max_utility_mean to 0.

    Returns:
        tuple: (areas, routerzero_areas) where areas is a list of lists and
            routerzero_areas is a list of trapezoidal areas for mean_script.
    """
    areas = [[] for _ in range(len(scripts))]
    routerzero_areas = []

    for seed_num in seeds:
        y_test = test_datasets[seed_num - start_seed]
        y_test_utility = -y_test.copy()
        max_utility_mean = 0 if zero_max_utility else np.max(y_test_utility, axis=1).mean()
        min_utility_mean = np.min(y_test_utility, axis=1).mean()

        for script_idx, script in enumerate(scripts):
            mem_path = f'{results_folder}/{script}/results{FOLDER_NAME}/utility_{which_exp}_{seed_num}.npy'
            memory_utilities = np.load(mem_path)

            points_x, points_y, last_x, first_x = _extract_pareto_points(
                memory_utilities, min_utility_mean, max_utility_mean
            )
            area, pareto_x, pareto_y = _pareto_area(
                points_x, points_y, first_x=first_x, last_x=last_x
            )
            areas[script_idx].append(area / (np.max(pareto_x) - np.min(pareto_x)))

            if script == 'mean_script':
                trap_area, px, py = _pareto_area_trapezoidal(
                    points_x, points_y, first_x=first_x, last_x=last_x
                )
                routerzero_areas.append(trap_area / (np.max(px) - np.min(px)))

    return areas, routerzero_areas


def _print_area_results(scripts, areas, routerzero_areas):
    """Print mean and std error for each script's Pareto area.

    Args:
        scripts: List of script names.
        areas: List of lists of area values per script.
        routerzero_areas: List of zero-router baseline areas.
    """
    for script_idx, script in enumerate(scripts):
        arr = np.array(areas[script_idx])
        print(f"{script}: {arr.mean():.8f} +/- {arr.std() / np.sqrt(len(arr)):.8f}")
    if routerzero_areas:
        arr = np.array(routerzero_areas)
        print(f"Zero router baseline: {arr.mean():.8f} +/- {arr.std() / np.sqrt(len(arr)):.8f}")


# ---------------------------------------------------------------------------
# Plotting functions
# ---------------------------------------------------------------------------

def _plot_pareto(memory_utilities, pareto_x, pareto_y, oracle_x, oracle_y,
                 min_utility_mean, max_utility_mean, title,
                 xlabel='Memory*Time (GB*s)', include_pass=False,
                 use_log_x=False):
    """Plot a Pareto deferral curve with oracle and zero-router baselines.

    Args:
        memory_utilities: Raw utility array for individual method markers.
        pareto_x: X coordinates of the Pareto frontier.
        pareto_y: Y coordinates of the Pareto frontier.
        oracle_x: X coordinates of the oracle frontier.
        oracle_y: Y coordinates of the oracle frontier.
        min_utility_mean: Lower normalization bound.
        max_utility_mean: Upper normalization bound.
        title: Plot title.
        xlabel: X-axis label.
        include_pass: If True, include the Laser Scanner method.
        use_log_x: If True, use log scale on x-axis.
    """
    fig, ax = plt.subplots(figsize=(8, 6.5))

    # Pareto and oracle curves
    ax.plot(pareto_x, pareto_y, color='#2B2B2B', linewidth=1.8,
            linestyle='-', zorder=3, label='Deferral curve')
    ax.plot(oracle_x, oracle_y, color='#6A0DAD', linewidth=1.8,
            linestyle='--', zorder=3, label='Oracle')

    # Zero router baseline: Pareto frontier of individual method points
    def _norm_y(method_idx):
        raw = memory_utilities[0, 0, method_idx]
        return (raw - min_utility_mean) / (max_utility_mean - min_utility_mean)

    n_methods = len(METHODS) + (1 if include_pass else 0)
    method_xs = np.array([memory_utilities[0, -1, j] for j in range(n_methods)])
    method_ys = np.array([_norm_y(j) for j in range(n_methods)])

    zr_x, zr_y = _pareto_frontier(method_xs, method_ys)

    # Extend rightward to cover the full deferral curve range
    end_x = max(pareto_x[-1], method_xs.max())
    if zr_x[-1] < end_x:
        zr_x = np.append(zr_x, end_x)
        zr_y = np.append(zr_y, zr_y[-1])

    # Draw segments between consecutive frontier points
    for i in range(len(zr_x) - 1):
        if use_log_x:
            xs = np.linspace(zr_x[i], zr_x[i + 1], 500)
            ys = zr_y[i] + (zr_y[i + 1] - zr_y[i]) * (xs - zr_x[i]) / (zr_x[i + 1] - zr_x[i])
            ax.plot(xs, ys, color='#808080', linewidth=1.8, linestyle='--',
                    zorder=2, label='Zero router' if i == 0 else None)
        else:
            ax.plot([zr_x[i], zr_x[i + 1]], [zr_y[i], zr_y[i + 1]],
                    color='#808080', linewidth=1.8, linestyle='--',
                    zorder=2, label='Zero router' if i == 0 else None)

    if use_log_x:
        ax.set_xscale('log')

    # Individual method markers
    method_names = list(METHODS)
    markers = list(METHOD_MARKERS)
    colors = list(METHOD_COLORS)
    if include_pass:
        method_names.append('Laser Scanner')
        markers.append('P')
        colors.append('#7B2D8E')

    for j, (name, marker, color) in enumerate(zip(method_names, markers, colors)):
        x = memory_utilities[0, -1, j]
        y = _norm_y(j)
        ax.scatter(x, y, color=color, marker=marker, s=60, zorder=5,
                   edgecolors='#2B2B2B', linewidths=0.6, label=name)

    ax.set_xlabel(xlabel, fontsize=20, fontweight='bold', labelpad=10)
    ax.set_ylabel('Normalized Utility', fontsize=20, fontweight='bold', labelpad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='in', labelsize=16)
    ax.grid(True, alpha=0.15, linewidth=0.5)
    ax.legend(fontsize=13, loc='best', framealpha=0.9, edgecolor='black',
              labelspacing=0.1)
    ax.set_title(title, fontsize=22, fontweight='bold', pad=20)
    plt.tight_layout(pad=0.5)
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Load test datasets across seeds (parallelized)
# ---------------------------------------------------------------------------

def _load_test_dataset(seed_num):
    """Load the test dataset y values for a single seed.

    Args:
        seed_num: Random seed.

    Returns:
        y_test array for the given seed.
    """
    data = np.load(data_path)
    y_all = data['y_all']
    all_embeddings = data['all_embeddings']
    all_metadata = data['all_metadata']
    _, _, test_dataset = k_fold_trainval_test_multi_object_styles(
        y_all, all_embeddings, all_metadata, k=5, test_split=0.2, seed=seed_num
    )
    return test_dataset.y


test_datasets_seed = Parallel(n_jobs=8)(
    delayed(_load_test_dataset)(seed) for seed in SEEDS
)
print(f"Loaded {len(test_datasets_seed)} test datasets")

In [ ]:
# ---------------------------------------------------------------------------
# Compute Pareto areas for all experiment types
# ---------------------------------------------------------------------------

SCRIPT_LABELS = {
    'scout_no_decouple': 'Scout_no_decouple',
    'scout_proper': 'Scout_proper',
    'knn_script': 'Knn',
    'linear_regression_script': 'Linear',
    'mlp_script': 'Mlp',
    'mf_script': 'MF',
    'oracle_script': 'Oracle',
}

DISPLAY_ORDER = [
    'mlp_script',
    'mf_script',
    'knn_script',
    'linear_regression_script',
    'scout_no_decouple',
    'scout_proper',
    '_zero_router',
    'oracle_script',
]

all_results = {}

for which_exp in EXPERIMENT_TYPES:
    zero_max = (which_exp == 'latencies')

    areas, routerzero_areas = _compute_areas_for_experiment(
        which_exp, test_datasets_seed, SCRIPTS, SEEDS,
        START_SEED, zero_max_utility=zero_max,
    )
    all_results[which_exp] = (areas, routerzero_areas)

    print(f'\n{which_exp}:')
    print('=' * 60)
    for key in DISPLAY_ORDER:
        if key == '_zero_router':
            arr = np.array(routerzero_areas)
            label = 'Zero Router'
        else:
            script_idx = SCRIPTS.index(key)
            arr = np.array(areas[script_idx])
            label = SCRIPT_LABELS[key]

        mean_val = arr.mean()
        std_err = arr.std() / np.sqrt(len(arr))

        if key == 'mean_script':
            print(f"{label:<25s} {mean_val:.4f}")
        else:
            print(f"{label:<25s} {mean_val:.4f}±{std_err:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Visualize Pareto frontiers for a single seed
# ---------------------------------------------------------------------------

PLOT_SEED = 199

PLOT_CONFIGS = {
    'memories': {
        'title': 'Memory versus Performance',
        'xlabel': 'Memory*Time (GB*s)',
        'include_pass': False,
        'use_log_x': False,
        'zero_max_utility': False,
    },
    'latencies': {
        'title': 'Latency versus Performance',
        'xlabel': 'Latency (s)',
        'include_pass': True,
        'use_log_x': True,
        'zero_max_utility': True,
    },
    'latency_memories': {
        'title': 'Latency*Memory versus Performance',
        'xlabel': 'Latency*Memory (GB*s)',
        'include_pass': False,
        'use_log_x': False,
        'zero_max_utility': False,
    },
}

y_test = test_datasets_seed[PLOT_SEED - START_SEED]
y_test_utility = -y_test.copy()

for which_exp, cfg in PLOT_CONFIGS.items():
    print(f"\n{'=' * 60}")
    print(f"Plotting: {which_exp} (seed={PLOT_SEED})")
    print('=' * 60)

    if cfg['zero_max_utility']:
        max_utility_mean = 0
    else:
        max_utility_mean = np.max(y_test_utility, axis=1).mean()
    min_utility_mean = np.min(y_test_utility, axis=1).mean()

    for script in SCRIPTS:
        if script != 'scout_proper':
            continue
        mem_path = f'{results_folder}/{script}/results{FOLDER_NAME}/utility_{which_exp}_{PLOT_SEED}.npy'
        memory_utilities = np.load(mem_path)

        points_x, points_y, last_x, first_x = _extract_pareto_points(
            memory_utilities, min_utility_mean, max_utility_mean
        )
        _, pareto_x, pareto_y = _pareto_area(
            points_x, points_y, first_x=first_x, last_x=last_x
        )

        # Load oracle for comparison
        oracle_path = f'{results_folder}/oracle_script/results{FOLDER_NAME}/utility_{which_exp}_{PLOT_SEED}.npy'
        oracle_utilities = np.load(oracle_path)
        oracle_px, oracle_py, oracle_lx, oracle_fx = _extract_pareto_points(
            oracle_utilities, min_utility_mean, max_utility_mean
        )
        _, oracle_pareto_x, oracle_pareto_y = _pareto_area(
            oracle_px, oracle_py, first_x=oracle_fx, last_x=oracle_lx
        )

        _plot_pareto(
            memory_utilities, pareto_x, pareto_y,
            oracle_pareto_x, oracle_pareto_y,
            min_utility_mean, max_utility_mean,
            title=cfg['title'],
            xlabel=cfg['xlabel'],
            include_pass=cfg['include_pass'],
            use_log_x=cfg['use_log_x'],
        )